# SAIGE-GENE SET-BASED ANALYSIS - PLINK FILE FOR VARIANCE RATIO
This script creates the PLINK file used for variance ratio estimation from a list of 10,000 randomly selected markers. Note that the file must contain at least 200 variants with MAC values falling within the specified categories. This PLINK file is used only for variance ratio estimation and contains randomly selected markers extracted from the LD-pruned PLINK file used for constructing the GRM and performing variance ratio estimation.

In [ ]:
# Load packages 
import os

# Set environment variable for workspace bucket 
my_bucket = os.getenv("WORKSPACE_BUCKET")

# Workspace bucket 
my_bucket

# COMMAND 

In [ ]:
%%writefile plink_markers.sh
#!/bin/bash

# Exit immediately if a command exits with a non-zero status
set -o errexit

# Treat unset variables as an error (prevents using undefined variables)
set -o nounset

# Loop through the populations
for pop in EUR; do
    echo "Processing population: ${pop}"
    
    # Copy new plink module into the plink2 
    cp "${plink2}" plink2

    # Grant permision to run plink2 in environment
    chmod +x plink2
    
    # ./plink2 : PLINK 2 executable used for genotype filtering and dataset generation
    # --bfile : input PLINK binary genotype prefix (bed/bim/fam dataset)
    # --keep : sample inclusion file specifying individuals retained for analysis
    # --extract : list of variant IDs to retain (here: 5000 marker subset for SAIGE)
    # --make-bed : outputs filtered dataset in PLINK binary format (bed/bim/fam)
    # --out : output file prefix for filtered genotype dataset used downstream in SAIGE
    
    # Extract markers from the large plink file
    ./plink2 --bfile "${genos}/arrays" \
             --keep "${cohort}/${pop}-like_exome_qc.txt" \
             --extract "${markers}/${pop}.5000.markerid.list" \
             --make-bed --out "${out_path}/${pop}_array.5000.forCate_vr"
        
done

In [ ]:
# Copy script to workspace bucket 
!gsutil cp plink_markers.sh {my_bucket}/data/scripts/                                    

In [ ]:
# Check script in workspace bucket  
!gsutil cat {my_bucket}/data/scripts/plink_markers.sh                               

# SUBMIT COMMAND 

In [ ]:
%%bash --out plink_markers 

#############################################################################
# SUBMIT DSUB COMMAND 
#############################################################################

# Source bash script 
source ~/aou_dsub.bash 

# Define the mount path for input PLINK files
mnt_path="/mnt/data/input/gs/fc-aou-datasets-controlled/v8/microarray/plink"

# --disk-size : size (GB) of attached persistent disk for runtime data and outputs
# --image : Docker image used to provide execution environment (Ubuntu base here)
# --logging : cloud storage path where job logs (stdout/stderr) are written
# --input plink2 : PLINK2 binary executable made available inside container
# --input bed : input PLINK BED file containing genotype data (SNP genotypes)
# --input bim : BIM file containing variant metadata (SNP IDs, positions, alleles)
# --input fam : FAM file containing sample information (individual IDs and phenotypes)
# --input-recursive cohort : directory of QC-filtered cohort sample lists used for inclusion filtering
# --input-recursive markers : directory containing marker ID lists used for variant extraction (e.g., 5000 SNPs)
# --output-recursive out_path : output directory for filtered PLINK datasets used downstream in SAIGE
# --env genos : environment variable pointing to mounted input genotype directory
# --script : bash script executed inside container performing PLINK2 filtering steps

aou_dsub \
  --disk-size 300 \
  --image ${ARTIFACT_REGISTRY_DOCKER_REPO}/ubuntu:latest \
  --logging "${WORKSPACE_BUCKET}/data/logging" \
  --input plink2="${WORKSPACE_BUCKET}/data/plink2_module/plink2" \
  --input bed="gs://fc-aou-datasets-controlled/v8/microarray/plink/arrays.bed" \
  --input bim="gs://fc-aou-datasets-controlled/v8/microarray/plink/arrays.bim" \
  --input fam="gs://fc-aou-datasets-controlled/v8/microarray/plink/arrays.fam" \
  --input-recursive cohort="${WORKSPACE_BUCKET}/data/wgs_qc/analysis_samples_wgs_qc/exome/" \
  --input-recursive markers="${WORKSPACE_BUCKET}/data/saige_step01/mark_list/" \
  --output-recursive out_path="${WORKSPACE_BUCKET}/data/saige_step01/variance_files/" \
  --env genos="${mnt_path}" \
  --script "${WORKSPACE_BUCKET}/data/scripts/plink_markers.sh"
  
#############################################################################
# END OF SCRIPT 
#############################################################################
